# Gauge-Equivariant CNN × ERA5 気温予測

Cohen et al. (2019) の U(1) ゲージ同変 CNN を正二十面体メッシュ上に実装し、
WeatherBench2 (ERA5) の 2m 気温データで 24 時間予測タスクを解く。

**Internet: ON / GPU: T4 推奨**

In [ ]:
!pip install gcsfs "zarr<3" xarray -q
!mkdir -p src

In [ ]:
%%writefile src/icosahedron.py
import numpy as np


def build_icosahedron():
    phi = (1 + np.sqrt(5)) / 2
    verts = np.array([
        [0,  1,  phi], [0, -1,  phi], [0,  1, -phi], [0, -1, -phi],
        [ 1,  phi, 0], [-1,  phi, 0], [ 1, -phi, 0], [-1, -phi, 0],
        [ phi, 0,  1], [-phi, 0,  1], [ phi, 0, -1], [-phi, 0, -1],
    ], dtype=np.float64)
    verts /= np.linalg.norm(verts, axis=1, keepdims=True)
    faces = np.array([
        [ 0,  4,  5], [ 0,  5,  9], [ 0,  9,  1], [ 0,  1,  8], [ 0,  8,  4],
        [ 4,  2,  5], [ 5,  2, 11], [ 5, 11,  9], [ 9, 11,  7], [ 9,  7,  1],
        [ 1,  7,  6], [ 1,  6,  8], [ 8,  6, 10], [ 8, 10,  4], [ 4, 10,  2],
        [ 3,  6,  7], [ 3,  7, 11], [ 3, 11,  2], [ 3,  2, 10], [ 3, 10,  6],
    ], dtype=np.int64)
    edge_set = set()
    for f in faces:
        for i in range(3):
            edge_set.add(tuple(sorted([int(f[i]), int(f[(i+1)%3])])))
    edges = np.array(sorted(edge_set), dtype=np.int64)
    return verts, faces, edges


def _subdivide_once(verts, faces):
    edge_map = {}
    new_verts = list(verts)
    new_faces = []
    def get_mid(i, j):
        key = (min(i,j), max(i,j))
        if key not in edge_map:
            mid = (new_verts[i] + new_verts[j]) / 2
            mid /= np.linalg.norm(mid)
            edge_map[key] = len(new_verts)
            new_verts.append(mid)
        return edge_map[key]
    for f in faces:
        v0,v1,v2 = int(f[0]),int(f[1]),int(f[2])
        m01,m12,m20 = get_mid(v0,v1),get_mid(v1,v2),get_mid(v2,v0)
        new_faces += [[v0,m01,m20],[v1,m12,m01],[v2,m20,m12],[m01,m12,m20]]
    return np.array(new_verts, dtype=np.float64), np.array(new_faces, dtype=np.int64)


def subdivide(verts, faces, level=1):
    for _ in range(level):
        verts, faces = _subdivide_once(verts, faces)
    return verts, faces


def compute_local_frames(vertices):
    V = len(vertices)
    frames = np.zeros((V, 2, 3))
    z_hat, x_hat = np.array([0.,0.,1.]), np.array([1.,0.,0.])
    for v, n in enumerate(vertices):
        e1 = np.cross(n, z_hat)
        if np.linalg.norm(e1) < 1e-8:
            e1 = np.cross(n, x_hat)
        e1 /= np.linalg.norm(e1)
        e2 = np.cross(n, e1)
        e2 /= np.linalg.norm(e2)
        frames[v,0], frames[v,1] = e1, e2
    return frames


def compute_connection_angles(vertices, edges, frames):
    angles = np.zeros((len(edges), 2))
    for e_idx, (v,w) in enumerate(edges):
        for direction, (src,dst) in enumerate([(v,w),(w,v)]):
            n,e1,e2 = vertices[src], frames[src,0], frames[src,1]
            d = vertices[dst] - vertices[src]
            d_tan = d - np.dot(d,n)*n
            angles[e_idx, direction] = np.arctan2(np.dot(d_tan,e2), np.dot(d_tan,e1))
    return angles


def build_adjacency(vertices, edges, angles):
    adj = [[] for _ in range(len(vertices))]
    for e_idx, (v,w) in enumerate(edges):
        adj[v].append((int(w), float(angles[e_idx,0])))
        adj[w].append((int(v), float(angles[e_idx,1])))
    return adj


def build_gauge_data_general(verts, faces, device=None):
    import torch
    frames = compute_local_frames(verts)
    edge_set = set()
    for f in faces:
        for i in range(3):
            edge_set.add(tuple(sorted([int(f[i]), int(f[(i+1)%3])])))
    edges  = np.array(sorted(edge_set), dtype=np.int64)
    angles = compute_connection_angles(verts, edges, frames)
    adj    = build_adjacency(verts, edges, angles)
    V, max_deg = len(verts), max(len(n) for n in adj)
    nb_idx_np  = np.zeros((V, max_deg), dtype=np.int64)
    nb_ang_np  = np.zeros((V, max_deg), dtype=np.float32)
    nb_mask_np = np.zeros((V, max_deg), dtype=np.float32)
    for v, nbrs in enumerate(adj):
        for k, (w, alpha) in enumerate(nbrs):
            nb_idx_np[v,k], nb_ang_np[v,k], nb_mask_np[v,k] = w, alpha, 1.0
    nb_idx  = torch.tensor(nb_idx_np,  dtype=torch.long)
    nb_ang  = torch.tensor(nb_ang_np,  dtype=torch.float32)
    nb_mask = torch.tensor(nb_mask_np, dtype=torch.float32)
    if device:
        nb_idx, nb_ang, nb_mask = nb_idx.to(device), nb_ang.to(device), nb_mask.to(device)
    return nb_idx, nb_ang, nb_mask, verts, faces


In [ ]:
%%writefile src/gauge_cnn.py
import numpy as np
import torch
import torch.nn as nn


class GaugeConv(nn.Module):
    def __init__(self, c_in, c_out, n_types, neighbor_idx, neighbor_angles):
        super().__init__()
        self.c_in, self.c_out, self.n_types = c_in, c_out, n_types
        self.register_buffer('neighbor_idx',    neighbor_idx)
        self.register_buffer('neighbor_angles', neighbor_angles)
        scale = 1.0 / np.sqrt(c_in * neighbor_idx.shape[1])
        self.weight_real = nn.Parameter(torch.randn(n_types, c_out, c_in) * scale)
        self.weight_imag = nn.Parameter(torch.randn(n_types, c_out, c_in) * scale)

    def forward(self, x, nb_mask=None):
        B, V, _ = x.shape
        K = self.neighbor_idx.shape[1]
        x_nb = x[:, self.neighbor_idx.reshape(-1), :].reshape(B, V, K, self.c_in)
        if nb_mask is not None:
            x_nb = x_nb * nb_mask[None, :, :, None]
        ns     = torch.arange(self.n_types, device=x.device, dtype=torch.float32)
        phases = ns[None,None,:] * self.neighbor_angles[:,:,None]
        cos_p, sin_p = torch.cos(phases), torch.sin(phases)
        agg_r = torch.einsum('bvkc,vkn->bvnc', x_nb, cos_p)
        agg_i = torch.einsum('bvkc,vkn->bvnc', x_nb, sin_p)
        out_r = (torch.einsum('noc,bvnc->bvno', self.weight_real, agg_r)
               - torch.einsum('noc,bvnc->bvno', self.weight_imag, agg_i))
        out_i = (torch.einsum('noc,bvnc->bvno', self.weight_real, agg_i)
               + torch.einsum('noc,bvnc->bvno', self.weight_imag, agg_r))
        return torch.stack([out_r, out_i], dim=-1)


class GaugeNorm(nn.Module):
    def __init__(self, n_types, c_out):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(n_types, c_out))
    def forward(self, x):
        norm  = x.norm(dim=-1)
        scale = torch.relu(norm + self.bias[None,None]) / norm.clamp(min=1e-8)
        return x * scale.unsqueeze(-1)


class InvariantPool(nn.Module):
    def forward(self, x):
        B, V, N, C, _ = x.shape
        return (x**2).sum(dim=-1).reshape(B, V, N*C)


class IcosGaugeCNNGeneral(nn.Module):
    def __init__(self, c_in, c_hidden, n_types, n_out, nb_idx, nb_ang, nb_mask, n_time_feats=0):
        super().__init__()
        self.register_buffer('nb_mask', nb_mask)
        self.n_time_feats = n_time_feats
        inv_dim = n_types * c_hidden
        self.conv1 = GaugeConv(c_in,    c_hidden, n_types, nb_idx, nb_ang)
        self.norm1 = GaugeNorm(n_types, c_hidden)
        self.pool1 = InvariantPool()
        self.ln1   = nn.LayerNorm(inv_dim)
        self.conv2 = GaugeConv(inv_dim, c_hidden, n_types, nb_idx, nb_ang)
        self.norm2 = GaugeNorm(n_types, c_hidden)
        self.pool2 = InvariantPool()
        self.ln2   = nn.LayerNorm(inv_dim)
        self.head  = nn.Linear(inv_dim + n_time_feats, n_out)

    def forward(self, x):
        if self.n_time_feats > 0:
            x_geo  = x[..., :-self.n_time_feats]
            t_feat = x[..., -self.n_time_feats:]
        else:
            x_geo  = x
        x = self.ln1(self.pool1(self.norm1(self.conv1(x_geo, self.nb_mask))))
        x = self.ln2(self.pool2(self.norm2(self.conv2(x,     self.nb_mask))))
        if self.n_time_feats > 0:
            x = torch.cat([x, t_feat], dim=-1)
        return self.head(x)

In [ ]:
%%writefile src/era5_loader.py
import numpy as np


def xyz_to_latlon(verts):
    lats = np.degrees(np.arcsin(np.clip(verts[:,2], -1., 1.)))
    lons = np.degrees(np.arctan2(verts[:,1], verts[:,0])) % 360.
    return lats, lons


def resample_to_mesh(field, src_lats, src_lons, tgt_verts):
    from scipy.interpolate import RegularGridInterpolator
    if src_lats[0] > src_lats[-1]:
        src_lats, field = src_lats[::-1], field[::-1,:]
    if src_lons.min() < 0:
        src_lons = src_lons % 360.
        order = np.argsort(src_lons)
        src_lons, field = src_lons[order], field[:, order]
    interp = RegularGridInterpolator((src_lats, src_lons), field,
                                     method='linear', bounds_error=False, fill_value=None)
    tgt_lats, tgt_lons = xyz_to_latlon(tgt_verts)
    return interp(np.column_stack([tgt_lats, tgt_lons]))


def load_weatherbench2(verts, variables=['2m_temperature'],
                       start='2018-01-01', end='2019-12-31',
                       pressure_level=None):
    import xarray as xr
    import gcsfs
    gcs = gcsfs.GCSFileSystem(token='anon')
    # 単一レベル・気圧面ともに同じ zarr に収録されている
    path = ('gs://weatherbench2/datasets/era5/'
            '1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr')
    print('WeatherBench2 に接続中...')
    store = gcs.get_mapper(path)
    try:
        ds = xr.open_zarr(store, consolidated=True)
    except Exception:
        ds = xr.open_zarr(store, consolidated=False)
    ds = ds.sel(time=slice(start, end))
    if pressure_level is not None and 'level' in ds.dims:
        ds = ds.sel(level=pressure_level, method='nearest')
    lats = ds['latitude'].values
    lons = ds['longitude'].values
    T, V, C = len(ds['time']), len(verts), len(variables)
    features = np.zeros((T, V, C), dtype=np.float32)
    for c, var in enumerate(variables):
        print(f"  '{var}' リサンプリング中 (T={T})...")
        da = ds[var]
        # dims が (time, longitude, latitude) の場合は転置
        if da.dims[1] == 'longitude':
            da = da.transpose('time', 'latitude', 'longitude')
        data = da.values  # (T, nlat, nlon)
        for t in range(T):
            features[t,:,c] = resample_to_mesh(data[t], lats, lons, verts)
    times = ds['time'].values
    ds.close()
    print(f'完了: shape={features.shape}')
    return features, times

## 設定

In [ ]:
CFG = {
    # ── メッシュ ──────────────────────────────────────────────────────
    'level'   : 2,

    # ── データ（WeatherBench2）────────────────────────────────────────
    'variables' : [
        '2m_temperature',
        'mean_sea_level_pressure',
        '10m_u_component_of_wind',
        '10m_v_component_of_wind',
    ],
    'wb2_start' : '2015-01-01',
    'wb2_end'   : '2022-12-31',

    # ── 予測設定 ──────────────────────────────────────────────────────
    'lag'     : 4,   # 6h × 4 = 24h先

    # ── 時刻特徴量（Option A）───────────────────────────────────────
    # hour_sin/cos + doy_sin/cos の4チャネルを入力に追加 → 日周サイクル学習
    'use_time_features': True,

    # ── 変数別損失重み ────────────────────────────────────────────────
    # 気温アノマリー予測のため気温の損失重みを引き上げる
    # アノマリーはスケールが小さく（std≈0.45）、放置すると学習が薄くなるため
    'loss_weights': [3.0, 1.0, 1.0, 1.0],

    # 気温チャンネルをアノマリー（変化量）予測に切り替え
    # 持続予測のベースラインが「変化なし=0」になりモデルが勝ちやすくなる
    'temp_anomaly': True,

    # ── モデル ────────────────────────────────────────────────────────
    'c_hidden': 32,
    'n_types' : 4,

    # ── 学習 ──────────────────────────────────────────────────────────
    'epochs'  : 100,
    'lr'      : 1e-3,
    'batch'   : 32,
}

## メッシュ構築

In [ ]:
import sys, os
sys.path.insert(0, 'src')

import numpy as np
import torch

from icosahedron import build_icosahedron, subdivide, build_gauge_data_general

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

v0, f0, _ = build_icosahedron()
verts, faces = subdivide(v0, f0, level=CFG['level'])
print(f"Icosahedron level {CFG['level']}: {len(verts)} 頂点")

nb_idx, nb_ang, nb_mask, verts, faces = build_gauge_data_general(verts, faces, device)
print(f'max_deg={nb_idx.shape[1]}  mask有効率={nb_mask.mean():.3f}')

## ERA5 データ取得（WeatherBench2）

In [ ]:
from era5_loader import load_weatherbench2
import pandas as pd

features, times = load_weatherbench2(
    verts,
    variables = CFG['variables'],
    start     = CFG['wb2_start'],
    end       = CFG['wb2_end'],
)
n_vars = len(CFG['variables'])
print(f'shape={features.shape}  期間: {times[0]} → {times[-1]}')

# ── 時刻特徴量を追加（Option A）─────────────────────────────────────────────
# 各時刻に対して [hour_sin, hour_cos, doy_sin, doy_cos] の4チャネルを生成し、
# 全頂点に同じ値を broadcast（時刻特徴は地点に依存しないため）。
if CFG['use_time_features']:
    times_pd = pd.to_datetime(times)
    hour = times_pd.hour.values
    doy  = times_pd.dayofyear.values

    hour_sin = np.sin(2 * np.pi * hour / 24).astype(np.float32)
    hour_cos = np.cos(2 * np.pi * hour / 24).astype(np.float32)
    doy_sin  = np.sin(2 * np.pi * doy  / 365).astype(np.float32)
    doy_cos  = np.cos(2 * np.pi * doy  / 365).astype(np.float32)

    time_feats = np.stack([hour_sin, hour_cos, doy_sin, doy_cos], axis=1)  # (T, 4)
    T, V = features.shape[:2]
    time_feats_expanded = np.broadcast_to(time_feats[:, None, :], (T, V, 4)).copy()

    features = np.concatenate([features, time_feats_expanded], axis=-1)
    n_input_channels = n_vars + 4
    print(f'時刻特徴量を追加: shape={features.shape}  (元の{n_vars}変数 + 時刻4ch)')
else:
    n_input_channels = n_vars


## データセット・分割

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MeshForecastDataset(Dataset):
    def __init__(self, features, n_vars, lag=4, temp_anomaly=True):
        self.X = torch.tensor(features[:-lag], dtype=torch.float32)
        y_abs = features[lag:, :, :n_vars].copy()
        if temp_anomaly:
            # 気温(ch0): 絶対値 → lagステップ分の変化量に変換
            # 持続予測のベースライン = 「変化なし = 0」になる
            y_abs[:, :, 0] -= features[:-lag, :, 0]
        self.y = torch.tensor(y_abs, dtype=torch.float32)
        self.temp_anomaly = temp_anomaly
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

# train で正規化（リーク防止）— 時刻特徴量は既に [-1, 1] なのでスキップしてもよい
T = len(features)
T_train = int(T * 0.7)
T_val   = int(T * 0.15)

# 元の n_vars 部分だけ標準化、時刻特徴量はそのまま
mean = np.zeros((1, 1, n_input_channels), dtype=np.float32)
std  = np.ones((1,  1, n_input_channels), dtype=np.float32)
mean[..., :n_vars] = features[:T_train, :, :n_vars].mean(axis=(0,1), keepdims=True)
std[...,  :n_vars] = features[:T_train, :, :n_vars].std(axis=(0,1),  keepdims=True).clip(min=1e-6)
features = (features - mean) / std

ds_train = MeshForecastDataset(features[:T_train],              n_vars=n_vars, lag=CFG['lag'], temp_anomaly=CFG['temp_anomaly'])
ds_val   = MeshForecastDataset(features[T_train:T_train+T_val], n_vars=n_vars, lag=CFG['lag'], temp_anomaly=CFG['temp_anomaly'])
ds_test  = MeshForecastDataset(features[T_train+T_val:],        n_vars=n_vars, lag=CFG['lag'], temp_anomaly=CFG['temp_anomaly'])

dl_train = DataLoader(ds_train, batch_size=CFG['batch'], shuffle=True,  num_workers=2)
dl_val   = DataLoader(ds_val,   batch_size=CFG['batch'], shuffle=False, num_workers=2)
dl_test  = DataLoader(ds_test,  batch_size=CFG['batch'], shuffle=False, num_workers=2)
print(f'Train:{len(ds_train)}  Val:{len(ds_val)}  Test:{len(ds_test)}')
print(f'入力ch:{n_input_channels}  出力ch:{n_vars}')

## モデル・学習

In [ ]:
import torch.nn as nn
from gauge_cnn import IcosGaugeCNNGeneral

model = IcosGaugeCNNGeneral(
    c_in         = n_vars,                                    # Gauge層には気象変数のみ
    c_hidden     = CFG['c_hidden'],
    n_types      = CFG['n_types'],
    n_out        = n_vars,
    nb_idx       = nb_idx,
    nb_ang       = nb_ang,
    nb_mask      = nb_mask,
    n_time_feats = 4 if CFG['use_time_features'] else 0,     # 時刻特徴量はhead直前でconcat
).to(device)
print(f'パラメータ数: {sum(p.numel() for p in model.parameters()):,}')

optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

# ── 変数別損失重み（Option C）────────────────────────────────────────────
# 持続予測が強い気温は重み下げ、力学変数（気圧・風速）を重視
loss_weights = torch.tensor(CFG['loss_weights'], device=device, dtype=torch.float32)
loss_weights = loss_weights / loss_weights.mean()  # 平均1に正規化
print(f'損失重み: {dict(zip(CFG["variables"], CFG["loss_weights"]))}')

def weighted_mse(pred, target):
    # pred, target: (B, V, n_vars)
    # 変数次元で重み付け → 平均
    sq = (pred - target) ** 2
    return (sq * loss_weights[None, None, :]).mean()

criterion = weighted_mse


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            if train: optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item()
    return total / len(loader)


best_val, best_state = float('inf'), None

for epoch in range(1, CFG['epochs']+1):
    tr = run_epoch(dl_train, train=True)
    vl = run_epoch(dl_val,   train=False)
    scheduler.step(vl)
    if vl < best_val:
        best_val, best_state = vl, {k: v.clone() for k,v in model.state_dict().items()}
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:>4}/{CFG["epochs"]} | train={tr:.4f}  val={vl:.4f}  (best={best_val:.4f})')

## テスト評価

In [ ]:
if best_state:
    model.load_state_dict(best_state)

# ── 変数ごとに別々に MSE を計算 ───────────────────────────────────────────────
# 注意: X は時刻特徴量込みなので、持続予測の比較では X[:, :, :n_vars] を使う
model.eval()
sse_var = np.zeros(n_vars)
sse_persist = np.zeros(n_vars)
count = 0
with torch.no_grad():
    for X, y in dl_test:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        X_state = X[:, :, :n_vars]  # 時刻特徴量を除いた元変数だけ
        sq_var     = (pred - y) ** 2            # (B, V, n_vars)
        sq_persist = (X_state - y) ** 2
        if CFG.get('temp_anomaly', False):
            # 気温: 持続予測 = 変化なし(0) → 誤差 = アノマリーy^2
            sq_persist[:, :, 0] = y[:, :, 0] ** 2
        sse_var     += sq_var.sum(dim=(0, 1)).cpu().numpy()
        sse_persist += sq_persist.sum(dim=(0, 1)).cpu().numpy()
        count += y.shape[0] * y.shape[1]

mse_var     = sse_var / count
mse_persist = sse_persist / count
test_loss   = mse_var.mean()
persist_loss = mse_persist.mean()
skill = 1.0 - test_loss / persist_loss

print(f'\n── テスト結果 ──')
print(f'  MSE (model, 全変数平均) : {test_loss:.4f}')
print(f'  MSE (持続予測, 全変数平均): {persist_loss:.4f}')
print(f'  Skill score (全変数平均) : {skill:.4f}  (>0 で持続予測より良い)')

# 変数ごとの RMSE を元のスケールに戻す
std_vals = std[..., :n_vars].reshape(-1)
var_names = CFG['variables']
print(f'\n  ── 変数ごとの性能 ──')
print(f'  {"変数":<35} {"RMSE":>10} {"Skill":>8} {"重み":>6}')
for i, (vname, s) in enumerate(zip(var_names, std_vals)):
    rmse_orig = np.sqrt(mse_var[i]) * float(s)
    skill_v = 1.0 - mse_var[i] / mse_persist[i]
    unit = 'K' if 'temperature' in vname else ('Pa' if 'pressure' in vname else 'm/s')
    w = CFG['loss_weights'][i]
    print(f'  {vname:<35} {rmse_orig:>7.3f} {unit:<3} {skill_v:>7.4f} {w:>6.2f}')

torch.save(model.state_dict(), 'gauge_cnn_era5.pt')
print('\n保存: gauge_cnn_era5.pt')